# Acciones por pantalla: exportar a `.litertlm` (FunctionGemma)

Convierte el modelo ya entrenado (`modelo.zip`, el contenido de `salida/modelo`) a `.litertlm` para el teléfono. **No entrena nada**: solo exporta.

Sigue el procedimiento del [notebook oficial de Google](https://github.com/google-gemini/gemma-cookbook/blob/main/FunctionGemma/%5BFunctionGemma%5DFinetune_FunctionGemma_270M_for_Mobile_Actions_with_Hugging_Face.ipynb): mismas instalaciones, mismos metadatos de FunctionGemma y mismas opciones de exportación.

**Antes de empezar**
1. Runtime > Change runtime type > GPU (una T4 alcanza).
2. Ten a mano `modelo.zip` (se sube en la celda 2, o se lee de Drive).
3. Ejecuta las celdas en orden. Si la celda 3 da `ModuleNotFoundError`, reinicia la sesión (Runtime > Restart session) y ejecuta desde la celda 2, **sin** volver a la 1.

In [ ]:
# 1. Dependencias (las de Google: sin TensorFlow y con ai-edge-torch reinstalado, así trae versiones que casan entre sí)
!pip uninstall -y -q tensorflow
!pip install -q ai-edge-torch-nightly --force-reinstall
!pip install -q ai-edge-litert-nightly

In [ ]:
# 2. Traer el modelo: subir modelo.zip, o leerlo de Drive (más rápido si pesa ~1 GB)
import glob
import os
import zipfile

USAR_DRIVE = False
RUTA_ZIP_DRIVE = '/content/drive/MyDrive/modelo.zip'

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP = RUTA_ZIP_DRIVE
else:
    from google.colab import files
    subido = files.upload()
    ZIP = next(iter(subido))

DESTINO = '/content/zip_modelo'
with zipfile.ZipFile(ZIP) as z:
    z.extractall(DESTINO)

configs = glob.glob(f'{DESTINO}/**/config.json', recursive=True)
assert configs, 'No se encontró config.json dentro del zip: ¿es el modelo fusionado (salida/modelo)?'
CHECKPOINT = os.path.dirname(configs[0])
print('Checkpoint:', CHECKPOINT)
print(sorted(os.listdir(CHECKPOINT)))

# El conversor necesita tokenizer.model; si el zip no lo trae, se baja del modelo base
if not os.path.exists(f'{CHECKPOINT}/tokenizer.model'):
    from huggingface_hub import hf_hub_download, notebook_login
    notebook_login()
    hf_hub_download('google/functiongemma-270m-it', 'tokenizer.model', local_dir=CHECKPOINT)
assert os.path.exists(f'{CHECKPOINT}/tokenizer.model'), 'Falta tokenizer.model'
print('OK: modelo y tokenizador listos')

## Convertir

`KV_CACHE_MAX_LEN = 2048`: el prompt más largo del entrenamiento mide 1.219 tokens (las 9 funciones), así que el 1024 de Google no alcanza. Si la conversión o el teléfono se quejan, prueba `1024` y reduce el prompt. `PREFILL_SEQ_LEN = 256` es el valor de Google.

El archivo sale como `acciones_q8_ekv{KV_CACHE_MAX_LEN}.litertlm`.

In [ ]:
# 3. Conversión a LiteRT-LM (cuantizada a int8)
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.layers import kv_cache
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig

PREFILL_SEQ_LEN = 256
KV_CACHE_MAX_LEN = 2048
SALIDA = '/content/litertlm'
os.makedirs(SALIDA, exist_ok=True)

# Metadatos de FunctionGemma (los del notebook de Google)
llm_metadata = r"""start_token: {
    token_ids: {
        ids: [ 2 ]
    }
}
stop_tokens: {
    token_str: "<end_of_turn>"
}
stop_tokens: {
    token_str: "<start_function_response>"
}
llm_model_type: {
    function_gemma: {}
}
"""
metadata_path = os.path.join(SALIDA, 'base_llm_metadata.textproto')
with open(metadata_path, 'w') as f:
    f.write(llm_metadata)

pytorch_model = gemma3.build_model_270m(CHECKPOINT)

export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

converter.convert_to_litert(
    pytorch_model,
    output_path=SALIDA,
    output_name_prefix='acciones',
    prefill_seq_len=PREFILL_SEQ_LEN,
    kv_cache_max_len=KV_CACHE_MAX_LEN,
    quantize='dynamic_int8',
    export_config=export_config,
    tokenizer_model_path=os.path.join(CHECKPOINT, 'tokenizer.model'),
    base_llm_metadata_path=metadata_path,
    output_format='litertlm',
)
!ls -lh {SALIDA}

## Descargar

El disco de Colab se borra al cerrar la sesión: baja el `.litertlm` antes de cerrar (pesa cerca de 300 MB). Para probarlo en el teléfono, Google usa la app **AI Edge Gallery**, importando el archivo desde Drive.

In [ ]:
# 4. Verificar y descargar
from google.colab import files

resultado = glob.glob(f'{SALIDA}/*.litertlm')
assert resultado, 'No se generó ningún .litertlm: revisa el error de la celda 3'
for f in resultado:
    print(f, f'{os.path.getsize(f) / 1e6:.0f} MB')
    files.download(f)

# Opcional: copiarlo también a Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/acciones && cp {SALIDA}/*.litertlm /content/drive/MyDrive/acciones/